## Install Dependencies

In [3]:
!pip install opencv-python matplotlib

In [5]:
#Only for macos
!pip install tensorflow-macos tensorflow-metal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.0/227.0 MB 1.1 MB/s eta 0:00:0000:0100:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.8/389.8 kB 1.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 1.3 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.17.1
    Uninstalling tensorboard-2.17.1:
      Successfully uninstalled tensorboard-2.17.1
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.17.0
    Uninstalling tensorflow-2.17.0:
      Successfully uninstalled tensorflow-2.17.0


## Import Dependencies

In [2]:
#Import standard dependencies
import cv2
import os
import random
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Import tensorflow dependencies - Functional API
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Conv2D, Dense, MaxPooling2D, Input, Flatten

## GPU Growth

In [4]:
# Avoid OOM errors by setting GPU Memory Consumption Growth
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

## Create Folder Structures

In [5]:
#Setup Paths
POS_PATH = os.path.join('data', 'positive')
NEG_PATH = os.path.join('data', 'negative')
ANC_PATH = os.path.join('data', 'anchor')

In [ ]:
# Make the directories
os.makedirs(POS_PATH)
os.makedirs(NEG_PATH)
os.makedirs(ANC_PATH)

## LFW Dataset

In [6]:
#Move LFW images to data/negative folder
for directory in os.listdir('lfw'):
    if directory.startswith('.'):
        continue  
    dir_path = os.path.join('lfw', directory)
    for file in os.listdir(dir_path):
        EX_PATH = os.path.join('lfw', directory, file)
        NEW_PATH = os.path.join(NEG_PATH, file)
        os.replace(EX_PATH, NEW_PATH)

## Collect Positive Anchor Classes

In [7]:
#Import uuid for generating unique names
import uuid

In [8]:
#Webcam connection establishment
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    
    #Cut down frame to 250X250
    frame = frame[250:250+250,530:530+250,:]
    
    #Collect Anchor image
    if cv2.waitKey(1) & 0XFF==ord('a'):
        imgname = os.path.join(ANC_PATH, '{}.jpg'.format(uuid.uuid1()))
        cv2.imwrite(imgname,frame)
        
    #Collect Positives
    if cv2.waitKey(1) & 0XFF==ord('p'):
        imgname = os.path.join(POS_PATH, '{}.jpg'.format(uuid.uuid1()))
        cv2.imwrite(imgname,frame)
        
    #Show image to screen
    cv2.imshow('Image Collection', frame)
    
    #Breaking 
    if cv2.waitKey(1) & 0XFF==ord('q'):
        break
        
#Release Webcam
cap.release()
#Close image show frame
if cv2.getWindowProperty('Image Collection', cv2.WND_PROP_VISIBLE) >= 1:
    cv2.destroyWindow('Image Collection')
cv2.waitKey(100)
cv2.destroyAllWindows()

## Load and Preprocess Images

In [10]:
anchor = tf.data.Dataset.list_files(ANC_PATH + '/*.jpg')
positive = tf.data.Dataset.list_files(POS_PATH + '/*.jpg')
negative = tf.data.Dataset.list_files(NEG_PATH + '/*.jpg')

In [13]:
dir_test = anchor.as_numpy_iterator()

In [14]:
print(dir.next())

b'data/anchor/2b4dc562-1c97-11f0-ba32-9e47e35cc52b.jpg'


In [34]:
def preprocess(img):
    img = tf.convert_to_tensor(img, dtype=tf.float32)  # ensure it's a Tensor
    img = tf.image.resize(img, (100, 100))             # resize if needed
    return img

In [33]:
trial_img = preprocess('data/anchor/2b4dc562-1c97-11f0-ba32-9e47e35cc52b.jpg')

In [19]:
anchor = anchor.map(preprocess)
positive = positive.map(preprocess)
negative = negative.map(preprocess)

In [21]:
positives = tf.data.Dataset.zip((anchor, positive, tf.data.Dataset.from_tensor_slices(tf.ones(len(anchor)))))
negatives = tf.data.Dataset.zip((anchor, negative, tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor)))))
data = positives.concatenate(negatives)

In [22]:
data

<_ConcatenateDataset element_spec=(TensorSpec(shape=(100, 100, None), dtype=tf.float32, name=None), TensorSpec(shape=(100, 100, None), dtype=tf.float32, name=None), TensorSpec(shape=(), dtype=tf.float32, name=None))>

In [23]:
samples = data.as_numpy_iterator()

In [29]:
samples.next()

(array([[[0.49534315, 0.4875    , 0.53063726],
         [0.5       , 0.49215686, 0.5352941 ],
         [0.5031863 , 0.49534315, 0.5384804 ],
         ...,
         [0.6933824 , 0.7286765 , 0.7247549 ],
         [0.70661765, 0.74583334, 0.74191177],
         [0.722549  , 0.7617647 , 0.7656863 ]],
 
        [[0.50784314, 0.49607843, 0.5392157 ],
         [0.49632353, 0.48455882, 0.5276961 ],
         [0.49754903, 0.48578432, 0.52892154],
         ...,
         [0.6848039 , 0.71911764, 0.71813726],
         [0.6938726 , 0.73308825, 0.73112744],
         [0.7       , 0.7382353 , 0.74019605]],
 
        [[0.49656862, 0.48480392, 0.52794117],
         [0.50514704, 0.49338236, 0.5365196 ],
         [0.5007353 , 0.48897058, 0.53210783],
         ...,
         [0.67769605, 0.70906866, 0.7169118 ],
         [0.6963235 , 0.7355392 , 0.73946077],
         [0.7029412 , 0.7411765 , 0.7372549 ]],
 
        ...,
 
        [[0.6875    , 0.6580882 , 0.6139706 ],
         [0.76593137, 0.73308825, 0.68921

In [25]:
def preprocess_twin(input_img, validation_img, label):
    return (preprocess(input_img), preprocess(validation_img), label)

In [36]:
res = preprocess_twin(*example)

In [37]:
len(res)

3

In [40]:
# Data loader pipeline
data = data.map(preprocess_twin)
data = data.cache()
data = data.shuffle(buffer_size = 1024)

In [55]:
#Training Partition
train_data = data.take(round(len(data)*.7))
train_data = train_data.batch(16)
train_data = train_data.prefetch(8)

In [56]:
train_samples = train_data.as_numpy_iterator()

In [57]:
train_sample = train_samples.next()

In [58]:
len(train_sample[0])

16

In [61]:
#Testing Partition
test_data = data.skip(round(len(data)*.7))
test_data = test_data.take(round(len(data)*.3))
test_data = test_data.batch(16)
test_data = test_data.prefetch(8)

## Model Engineering

In [65]:
def make_embedding():
    input = Input(shape=(100,100,3),name='input_image')
    #First Block
    c1 = Conv2D(64, (10, 10), activation='relu')(input)
    m1 = MaxPooling2D(64, (2, 2), padding='same')(c1)

    #Second Block
    c2 = Conv2D(128, (7, 7), activation='relu')(m1)
    m2 = MaxPooling2D(64, (2, 2), padding='same')(c2)

    #Third Block
    c3 = Conv2D(128, (4, 4), activation='relu')(m2)
    m3 = MaxPooling2D(64, (2, 2), padding='same')(c3)

    #Final Embedding Block
    c4 = Conv2D(256, (4, 4), activation='relu')(m3)
    f1 = Flatten()(c4)
    d1 = Dense(4096, activation='sigmoid')(f1)
    
    return Model(inputs=[input], outputs=[d1], name='embedding')

In [66]:
model = make_embedding()

In [67]:
model.summary()

Model: "embedding"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 100, 100, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 91, 91, 64)     │        19,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 46, 46, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 40, 40, 128)    │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 20, 20, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 17, 17, 128)    │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 9, 9, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 6, 6, 256)      │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4096)           │    37,752,832 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,960,448 (148.62 MB)

 Trainable params: 38,960,448 (148.62 MB)

 Non-trainable params: 0 (0.00 B)

In [70]:
## Distance Layer
class L1Dist(Layer):
    def __init__(self, **kwargs):
        super().__init__

    def call(self, input_embedding, validation_embedding):
        return tf.math.abs(input_embedding - validation_embedding)